<img src="https://raw.githubusercontent.com/AliAkrami1375/dcho/main/assets/logo.png" width="90">

# dcho — training on Colab

Persian text-to-speech. Trains here, checkpoints to Google Drive and the
Hugging Face Hub.

**This notebook is deliberately thin.** Every cell below is a few lines
that call into the repository, because notebook cells do not update: a
notebook opened last week keeps running last week's logic even though the
first thing it does is pull fresh code. Keeping the training loop in
`dcho/train/session.py` means a fix reaches you on the next run, whatever
copy of this notebook you opened.

**Before running:**

1. `Runtime → Change runtime type → GPU`
2. Key icon 🔑 in the left sidebar → secret named `HF_TOKEN`, write access
3. Run every cell in order. Cell 3 will ask permission to mount Drive.

**If the session dies, just run it again from the top.** It picks up from
the last checkpoint.

## 1 · Hardware

In [ ]:
import subprocess, torch

print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                     capture_output=True, text=True).stdout.strip() or "NO GPU")
if not torch.cuda.is_available():
    raise SystemExit("No GPU. Runtime -> Change runtime type -> GPU, then rerun.")

name = torch.cuda.get_device_name()
RELATIVE = {"T4": 0.37, "L4": 1.00, "A100": 2.50, "V100": 0.75, "P100": 0.45}
speed = next((v for k, v in RELATIVE.items() if k in name), 0.6)
print(f"{name}  ~{speed:.2f}x an L4")

## 2 · Code and credentials

In [ ]:
%%capture
!pip -q install soundfile "huggingface_hub>=0.28"
!git clone -q https://github.com/AliAkrami1375/dcho.git /content/dcho 2>/dev/null || true
!cd /content/dcho && git fetch -q origin && git reset -q --hard origin/main

In [ ]:
import os, sys

sys.path.insert(0, "/content/dcho")
os.chdir("/content/dcho")

from google.colab import userdata
os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from huggingface_hub import HfApi
print("signed in as", HfApi(token=os.environ["HF_TOKEN"]).whoami()["name"])
!git log --oneline -1

## 3 · Google Drive

Everything under `/content` is destroyed when the runtime is reclaimed, and
when that happens the process is killed outright — no exception, no
`finally`, nothing gets a chance to save. Drive is a mounted filesystem
that outlives the runtime, so a checkpoint written there is simply already
safe. Without it a disconnect costs however long since the last upload.

In [ ]:
from dcho.train.session import mount_drive

DRIVE = mount_drive()          # asks for permission the first time
print("checkpoints will live in:", DRIVE or "nowhere persistent — see the warning below")
if DRIVE is None:
    print("Drive is not mounted. Training still works, but a disconnect will")
    print("cost everything since the last Hub upload.")

## 4 · Train

One call. It downloads the packed dataset, builds the model, resumes from
whatever checkpoint exists (Drive first, then the Hub), trains, and writes
a checkpoint on a timer — to Drive every 10 minutes and to the Hub every
hour. A disconnect therefore costs at most ten minutes of work.

`micro` is the cheap probe: it tells you whether alignment forms and the
loss curve is healthy. It is not meant to sound good. `base` is the real
model, and on a T4 it needs `batch_size` halved in `configs/base.json`.

In [ ]:
from dcho.train.session import run_session

result = run_session(
    config_name="micro",                 # micro | nano | base
    dataset="DibaAi/dcho-tier-a",        # 25,654 clips, 65.3 h
    run_repo="DibaAi/dcho-run-micro",
    max_steps=30_000,
    drive_dir=DRIVE,
    drive_minutes=10,                    # how much a disconnect can cost
    hub_minutes=60,
    repo_root="/content/dcho",
    device="cuda",
)

print()
print(result["outcome"], "at step", result["step"], f"lr {result['lr']:.3e}")
if result["detail"]:
    print(result["detail"])

## 5 · Listen

Alignment has to lock in before this is speech. Before roughly step 10,000
expect noise — that is what an untrained model sounds like, not a fault.
Watch `align_H` in the log above: it rises, then falling is the sign that
alignment is forming.

In [ ]:
import torch
from IPython.display import Audio, display

from dcho.text.frontend import Frontend

trainer = result["trainer"]
SAMPLE_RATE = trainer.data_cfg["sampling_rate"]
net = trainer.net_g.eval()
fe = Frontend()

# A real voice from the corpus, rather than an arbitrary direction in
# speaker space that no speaker occupies.
import numpy as np, pyarrow.parquet as pq, glob
row = pq.read_table(sorted(glob.glob("/content/data/**/*.parquet", recursive=True))[0],
                    columns=["speaker_vector"]).slice(0, 1).to_pydict()
speaker = torch.from_numpy(
    np.frombuffer(row["speaker_vector"][0], dtype=np.float16).astype(np.float32)
).unsqueeze(0).cuda()

for text in ["سلام، حال شما چطور است؟",
             "زبان فارسی یکی از زبان‌های کهن جهان است."]:
    out = fe(text)
    ids = torch.tensor([out.ids]).cuda()
    with torch.no_grad():
        audio, *_ = net.infer(ids, torch.tensor([ids.shape[1]]).cuda(), sid=speaker)
    print(text)
    print("  ", out.phonemes)
    display(Audio(audio.squeeze().cpu().numpy(), rate=SAMPLE_RATE))

---

### If the session dies

Run the notebook again from the top. Cell 2 resets the repository to the
newest code, cell 3 remounts Drive, and cell 4 finds the checkpoint and
continues. The line it prints — `resumed from drive at step N, lr ...` —
is the confirmation. If that learning rate ever reads `2.000e-04` after a
resume, something has gone wrong; say so.

### Keeping a free session alive

Leave the tab open and visible. Colab reclaims idle runtimes and a
backgrounded tab counts as idle. Colab Pro adds background execution,
which is the only reliable way to run unattended for long.

### Watching from outside

`latest.pth` and the periodic `snap_*.G.pth` land in the run repository on
the Hub, so progress is visible without touching this notebook.